# Test de NMF pour l'analyse de topic

## Chargement des librairies nécessaires

In [33]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF


In [34]:
df=pd.read_csv("corpus_zola_nettoye.csv") # Chargement du DataFrame nettoyé à partir du fichier CSV

In [49]:
def segmenter_texte(text, taille=1200): # On segmente le texte en segments de 300 mots
    mots = text.split() # On divise le texte en mots
    segments = [] # Liste pour stocker les segments de texte
    for i in range(0, len(mots), taille): # On boucle sur les mots par pas de "taille" (300 mots)
        segment = " ".join(mots[i:i+taille]) # On crée un segment en joignant les mots du segment
        if len(segment.split()) >= 100:  # On vérifie que le segment contient au moins 100 mots pour éviter les segments trop courts
            segments.append(segment) # On ajoute le segment à la liste des segments
    return segments # On retourne la liste des segments de texte

lignes = [] # Liste pour stocker les segments de texte avec leurs titres et IDs

for _, row in df.iterrows():
    titre = row["nom_fichier"] # On utilise le nom du fichier comme titre
    texte = row["texte_nettoye"] # On segmente le texte en segments de 300 mots
    segments = segmenter_texte(texte) # On ajoute chaque segment à la liste avec son titre et son ID
    
    for j, seg in enumerate(segments):
        
        
        lignes.append({ 
            "titre": titre, # Titre du texte (nom du fichier)
            "segment_id": j, # ID du segment (index du segment dans le texte) 
            "texte_segment": seg # Contenu du segment de texte
        })

df_segments = pd.DataFrame(lignes) # Création d'un DataFrame à partir de la liste de segments

df_segments

,titre,segment_id,texte_segment
0,1893_20_Le_docteur_Pascal._clean.txt,0,chaleur ardent après-midi juillet salle volet ...
1,1893_20_Le_docteur_Pascal._clean.txt,1,papier pupitre coup œil fauteuil muet sourd on...
2,1893_20_Le_docteur_Pascal._clean.txt,2,porte entrée chemin minute premier maison aire...
3,1893_20_Le_docteur_Pascal._clean.txt,3,volonté réel intention vie monde volonté force...
4,1893_20_Le_docteur_Pascal._clean.txt,4,clair vide beauté inquiétant ombre mort cervea...
...,...,...,...
903,1894_1_Lourdes._clean.txt,34,pâle oeil luisant poussée cohue hâte fou raiso...
904,1894_1_Lourdes._clean.txt,35,miracle formelle promesse miracle bout journée...
905,1894_1_Lourdes._clean.txt,36,tête pâle lueur dansant lampe fond compartimen...
906,1894_1_Lourdes._clean.txt,37,oubli total monde désert lointain jour peine a...


In [50]:
df_segments["nb_tokens_nettoyes"] = df_segments["texte_segment"].apply(lambda x: len(x.split())) # Calcul du nombre de tokens nettoyés pour chaque segment
df_segments["nb_tokens_nettoyes"].describe() # Affichage des statistiques descriptives du nombre de tokens nettoyés par segment (moyenne, écart-type, min, max, etc.)

count     908.000000
mean     1185.819383
std        98.428369
min       171.000000
25%      1200.000000
50%      1200.000000
75%      1200.000000
max      1200.000000
Name: nb_tokens_nettoyes, dtype: float64

In [51]:
df_segments

,titre,segment_id,texte_segment,nb_tokens_nettoyes
0,1893_20_Le_docteur_Pascal._clean.txt,0,chaleur ardent après-midi juillet salle volet ...,1200
1,1893_20_Le_docteur_Pascal._clean.txt,1,papier pupitre coup œil fauteuil muet sourd on...,1200
2,1893_20_Le_docteur_Pascal._clean.txt,2,porte entrée chemin minute premier maison aire...,1200
3,1893_20_Le_docteur_Pascal._clean.txt,3,volonté réel intention vie monde volonté force...,1200
4,1893_20_Le_docteur_Pascal._clean.txt,4,clair vide beauté inquiétant ombre mort cervea...,1200
...,...,...,...,...
903,1894_1_Lourdes._clean.txt,34,pâle oeil luisant poussée cohue hâte fou raiso...,1200
904,1894_1_Lourdes._clean.txt,35,miracle formelle promesse miracle bout journée...,1200
905,1894_1_Lourdes._clean.txt,36,tête pâle lueur dansant lampe fond compartimen...,1200
906,1894_1_Lourdes._clean.txt,37,oubli total monde désert lointain jour peine a...,1200


In [54]:
vectorizer = TfidfVectorizer( 
    min_df=6,
    max_df=0.8
    )
# min_df=5 : Ignorer les termes qui apparaissent dans moins de 3 segments, car ils sont considérés comme peu informatifs.
# max_df=0.8 : Ignorer les termes qui apparaissent dans plus de 80% des segments, car ils sont trop fréquents et ne contribuent pas à différencier les segments.

X = vectorizer.fit_transform(df_segments["texte_segment"])  
#Appliquer le TF-IDF vectorizer sur les segments de texte nettoyés pour obtenir une matrice de caractéristiques (termes pondérés par leur importance dans les segments).

X.shape

(908, 8373)

In [55]:
def afficher_topics(model, feature_names, n_top_words=8):
    for topic_idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]]
        print(f"Topic {topic_idx+1} : {' | '.join(top_words)}")



feature_names = vectorizer.get_feature_names_out()

#for k in  [8, 10, 12, 15, 18 ]: # On teste différents nombres de topics (de 5 à 15) pour trouver le nombre optimal de topics pour notre analyse.
    
    #print(f"\n NMF avec {k} topics") 
nmf = NMF(n_components=12, 
          random_state=42,
          init="nndsvda",
          solver="cd",
          max_iter=500,
          
            )
W = nmf.fit_transform(X)
afficher_topics(nmf, feature_names)

print("Erreur :", nmf.reconstruction_err_)




Topic 1 : monsieur | franc | dame | salon | vendeur | argent | affaire | rue
Topic 2 : travail | usine | bonheur | œuvre | ouvrier | abîme | fils | père
Topic 3 : prussien | soldat | armée | général | empereur | cheval | canon | route
Topic 4 : abbé | prêtre | curé | trouche | église | monsieur | jardin | sous
Topic 5 : insurgé | félicité | silvère | ville | fusil | national | fine | granou
Topic 6 : cardinal | pape | palais | eminence | boccanera | livre | romain | dario
Topic 7 : coupeau | rue | boutique | zingueur | blanchisseur | maman | vin | lorilleu
Topic 8 : fosse | maheude | coron | mineur | berline | grève | camarade | porion
Topic 9 : grotte | saint | malade | miracle | sœur | pèlerin | wagon | prêtre
Topic 10 : monsieur | docteur | mère | chambre | deberl | lit | malade | médecin
Topic 11 : instituteur | école | frère | père | élève | modèle | vérité | église
Topic 12 : amour | amant | chambre | arbre | soleil | pensée | lèvre | ciel
Erreur : 26.283927911273437
